# cli

> The CLI driver — the decomposition core's first (and currently only) frontend.

Ships in-package as the `cjm-transcript-decomp-core` console script so the driver can never skew from the core. GUI presentation drivers come later and consume the same `pipeline` module; they never reimplement it (CLI-first / headless-core principle).

Prerequisite runtime (once, from the repo root):
```bash
cjm-ctl --cjm-config cjm.yaml setup-runtime
cjm-ctl --cjm-config cjm.yaml install-all --plugins plugins_test.yaml --force
```
Then decompose a transcription-core run manifest:
```bash
cjm-transcript-decomp-core run path/to/transcription-run.json --yes
# GPU runs: opt into CR-7 GPU subtree attribution
cjm-transcript-decomp-core run run.json --yes --sysmon-plugin cjm-system-monitor-nvidia
```

In [ ]:
#| default_exp cli

In [ ]:
#| export
import argparse
import asyncio
import logging
from pathlib import Path
from typing import List, Optional

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue

from cjm_transcript_decomp_core.models import DecompConfig
from cjm_transcript_decomp_core.pipeline import run_decomp, load_source_manifest

logger = logging.getLogger(__name__)

In [ ]:
#| export
def build_parser() -> argparse.ArgumentParser:  # Configured CLI parser
    """Build the CLI parser (subcommands: run)."""
    parser = argparse.ArgumentParser(
        prog="cjm-transcript-decomp-core",
        description="Headless transcript decomposition: VAD + forced alignment -> graph spine.",
    )
    sub = parser.add_subparsers(dest="command", required=True)

    run = sub.add_parser("run", help="Decompose a transcription-core run manifest")
    run.add_argument("manifest", help="Transcription-core run manifest JSON (the source to decompose)")
    run.add_argument("--manifests-dir", default=".cjm/manifests", help="Capability manifests directory")
    run.add_argument("--vad-plugin", default="cjm-media-plugin-silero-vad", help="VAD capability name")
    run.add_argument("--fa-plugin", default="cjm-transcription-plugin-qwen3-forced-aligner", help="Forced-alignment capability name")
    run.add_argument("--graph-plugin", default="cjm-graph-plugin-sqlite", help="Graph-storage capability name")
    run.add_argument("--sysmon-plugin", default=None, help="MonitorPlugin capability for GPU subtree attribution (CR-7); loaded first; default: no monitor")
    run.add_argument("--language", default="English", help="Forced-alignment language")
    run.add_argument("--force", action="store_true", help="Bypass capability-side caches (VAD + FA)")
    run.add_argument("-y", "--yes", action="store_true", help="Auto-accept HITL seams (headless mode)")
    run.add_argument("--output", default=None, help="Decomp-manifest output path (default: runs/<run_id>.json)")
    run.add_argument("-v", "--verbose", action="store_true", help="DEBUG-level logging")
    return parser

In [ ]:
#| export
def load_capabilities(
    manager: PluginManager,   # Freshly constructed manager
    instance_ids: List[str],  # Capability names to load (default instances), in order
) -> None:
    """Discover manifests + load each requested capability (default instance)."""
    manager.discover_manifests()
    discovered = {m.name: m for m in manager.discovered}
    for iid in instance_ids:
        meta = discovered.get(iid)
        if meta is None:
            raise SystemExit(
                f"capability {iid!r} not found in manifests "
                f"(discovered: {sorted(discovered)}) — run cjm-ctl install-all first"
            )
        if not manager.load_plugin(meta):
            raise SystemExit(f"failed to load capability {iid!r}")
        logger.info(f"loaded {iid}")

In [ ]:
#| export
async def run_command(
    args: argparse.Namespace,  # Parsed CLI arguments for the `run` subcommand
) -> int:  # Process exit code (0 = all sources decomposed + committed)
    """Execute the `run` subcommand: decompose a transcription manifest into a graph spine."""
    manifest_path = str(Path(args.manifest).resolve())
    if not Path(manifest_path).exists():
        raise SystemExit(f"source manifest not found: {manifest_path}")

    cfg = DecompConfig(
        vad_plugin=args.vad_plugin,
        fa_plugin=args.fa_plugin,
        graph_plugin=args.graph_plugin,
        language=args.language,
        force=args.force,
        assume_yes=args.yes,
    )

    # CR-7 GPU subtree attribution is opt-in: --sysmon-plugin threads the monitor
    # name into BOTH the manager and the queue; the monitor loads FIRST so GPU
    # capabilities' samples record gpu_memory_mb_peak (voxtral-vllm e2e pattern).
    manager = PluginManager(
        search_paths=[Path(args.manifests_dir)],
        sysmon_plugin_name=args.sysmon_plugin,
    )
    instance_ids = [cfg.vad_plugin, cfg.fa_plugin, cfg.graph_plugin]
    load_order = ([args.sysmon_plugin] if args.sysmon_plugin else []) + instance_ids
    load_capabilities(manager, load_order)

    queue = JobQueue(deps=manager, sysmon_plugin_name=args.sysmon_plugin)
    await queue.start()
    try:
        manifest = await run_decomp(manager, queue, cfg, manifest_path)
    finally:
        await queue.stop()
        for iid in reversed(load_order):  # Reverse load order; the monitor unloads last
            try:
                manager.unload_plugin(iid)
            except Exception as e:  # Best-effort teardown; never mask the run's outcome
                logger.warning(f"unload {iid} failed: {e}")

    out = Path(args.output) if args.output else Path("runs") / f"{manifest.run_id}.json"
    manifest.save(out)
    n_sources = len(load_source_manifest(manifest_path).get("sources", []) or [])
    n_docs = len(manifest.documents)
    n_segs = sum(d.segment_count for d in manifest.documents)
    print(f"decomp manifest: {out}")
    print(f"documents committed: {n_docs}/{n_sources}  segment nodes: {n_segs}")
    return 0 if n_docs == n_sources else 1

In [ ]:
#| export
def main(
    argv: Optional[List[str]] = None,  # Argument list override (None = sys.argv)
) -> int:  # Process exit code
    """CLI entry point (console script: `cjm-transcript-decomp-core`)."""
    args = build_parser().parse_args(argv)
    logging.basicConfig(
        level=logging.DEBUG if args.verbose else logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s :: %(message)s",
    )
    if args.command == "run":
        return asyncio.run(run_command(args))
    raise SystemExit(f"unknown command: {args.command}")

In [ ]:
# CLI parser smoke checks (no plugins involved)
p = build_parser()
args = p.parse_args(["run", "m.json", "--yes", "--language", "English"])
assert args.command == "run"
assert args.manifest == "m.json"
assert args.yes is True
assert args.fa_plugin == "cjm-transcription-plugin-qwen3-forced-aligner"
assert args.graph_plugin == "cjm-graph-plugin-sqlite"
assert args.sysmon_plugin is None

args = p.parse_args(["run", "m.json", "--sysmon-plugin", "cjm-system-monitor-nvidia"])
assert args.sysmon_plugin == "cjm-system-monitor-nvidia"
print("cli parser checks OK")

cli parser checks OK
